# buffer-copy_-inplace — worked example 1: Update a BatchNorm running_var buffer in place with copy_

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `buffer-copy_-inplace`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A registered `nn.Module` buffer (like BatchNorm's `running_var`) is referenced by the module's internal buffer dict. To update it you must overwrite the *existing* tensor's storage with `tensor.copy_(new_value)`. Reassigning the Python variable would leave the module still pointing at the old tensor, so `copy_` is the correct mechanic.

## Worked solution

**Goal.** Update `running_var` to the EMA `(1 - momentum) * running_var + momentum * batch_var` while keeping `id(running_var)` and its storage identical.

**Step 1 — compute the new value in a fresh tensor.** `new_value = (1 - momentum) * running_var + momentum * batch_var`. This allocates a brand-new tensor; it does NOT touch the buffer yet. We compute first so the math is clear and so `copy_` has a fully-formed source.

**Step 2 — write it into the buffer's storage.** `running_var.copy_(new_value)` copies element-by-element into `running_var`'s existing memory. The Python object `running_var` is unchanged — same `id`, same `data_ptr` — only its contents change. This is exactly why the module's registered-buffer link survives.

**Why not reassign?** `running_var = new_value` would rebind the local name to a different tensor. The caller (and the module) still hold the old one, so the update is silently lost. `copy_` mutates in place, which is what buffer updates require.

In [ ]:
def update_running_var(running_var: Tensor, batch_var: Tensor, momentum: float) -> None:
    new_value = (1 - momentum) * running_var + momentum * batch_var
    running_var.copy_(new_value)

t.manual_seed(0)
running_var = t.ones(4)
batch_var = t.tensor([2.0, 4.0, 6.0, 8.0])
before = id(running_var)
update_running_var(running_var, batch_var, momentum=0.1)
print("id preserved:", id(running_var) == before)
print("running_var:", running_var)